# Getting Started with Award Pynder

This notebook demonstrates the basic usage of `award_pynder`, a Python package for searching grant and award databases across multiple funding agencies.

This is the Python equivalent of the [awardFindR R vignette](https://github.com/ropensci/awardFindR/blob/master/vignettes/awardFindR.Rmd).

In [ ]:
from award_pynder import search_awards, SOURCE_REGISTRY

# List all available sources
print("Available sources:")
for name, cls in sorted(SOURCE_REGISTRY.items()):
    print(f"  {name:15s} -> {cls.__name__}")

## Simple keyword search, single source

Search NSF for grants matching a keyword within a date range:

In [ ]:
nsf = search_awards(
    keywords="illicit",
    sources=["nsf"],
    from_date="2023-01-01",
)
print(f"Found {len(nsf)} grants")
nsf.info()
nsf.head()

## Multiple sources and keywords with a date range

Search across both NSF and NIH with multiple keywords:

In [ ]:
results = search_awards(
    keywords=["ontological", "audio recordings"],
    sources=["nsf", "nih"],
    from_date="2018-01-01",
    to_date="2018-02-01",
)

print(f"Found {len(results)} grants total\n")

# Count by source
print("Grants by source:")
print(results["source"].value_counts().to_string())

# Unique keywords matched
print(f"\nUnique queries: {results['query'].unique().tolist()}")

## Loading keywords from a file

For larger searches, you can pass a path to a CSV or text file with one keyword per line:

In [ ]:
import tempfile
import os

# Create a temporary keywords file
keywords_file = tempfile.NamedTemporaryFile(
    mode="w", suffix=".csv", delete=False
)
keywords_file.write("qualitative data\nqualitative analysis\ncase study\ncase studies\n")
keywords_file.close()

print(f"Keywords file: {keywords_file.name}")
print(f"Contents:")
print(open(keywords_file.name).read())

# Pass file path as keywords argument
file_results = search_awards(
    keywords=keywords_file.name,
    sources=["nsf"],
    from_date="2023-06-01",
    to_date="2023-07-01",
)
print(f"\nFound {len(file_results)} grants")
print(f"Unique queries: {file_results['query'].unique().tolist()}")

os.unlink(keywords_file.name)

## Using the CLI

Award Pynder also provides a command-line interface:

```bash
# Search NSF for "climate change" grants
award-pynder "climate change" --sources nsf --from-date 2020-01-01

# Multiple keywords, save to CSV
award-pynder "machine learning" "data science" -s nsf,nih -o results.csv

# Verbose mode with progress bars
award-pynder "qualitative" -s nsf -v
```